In [ ]:
%pip install ollama

In [ ]:
%pip install requests tqdm

In [5]:
import pandas as pd
import requests
import concurrent.futures
from tqdm.auto import tqdm
import re

MODEL_NAME = "gemma3:12b"
INPUT_CSV = '/home/greg/issuebench/2_final_dataset/combined_prompts_issues_with_topics.csv' # "/data/gregIB/issuebench/2_final_dataset/combined_prompts_issues_with_topics.csv"
SAFE_MODEL_NAME = re.sub(r'[:/\\]', '-', MODEL_NAME)
OUTPUT_CSV = f"/home/greg/issuebench/3_experiments/2_inference/completions/020925_{SAFE_MODEL_NAME}_completions.csv"

# DO NOT CHANGE
TEMPERATURE = 1
MAX_TOKENS = 514
MAX_WORKERS = 6

# CHANGE IF NEEDED
TEST_MODE = True  # Set to False for full processing
TEST_SAMPLE_SIZE = 50  # Number of rows to process in test mode

def process_row(args):
    index, row, model, temp, tokens = args
    payload = {
        "model": model,
        "prompt": row['prompt_text'],
        "think": False, # for thinking models --> will not output <think> , </think> strings [040925]
        "options": {
            "temperature": temp,
            "num_predict": tokens
        },
        "stream": False
    }
    
    try:
        response = requests.post("http://localhost:11434/api/generate", 
                               json=payload, 
                               timeout=300)
        response.raise_for_status()
        return (index, response.json()['response'])
    except Exception as e:
        return (index, f"ERROR: {str(e)}")

# Read and prepare data
df = pd.read_csv(INPUT_CSV)
if TEST_MODE:
    df = df.head(TEST_SAMPLE_SIZE)

df['model'] = MODEL_NAME

# Prepare arguments for parallel processing
tasks = [(i, row, MODEL_NAME, TEMPERATURE, MAX_TOKENS) 
         for i, row in df.iterrows()]

# parallel processing
results = {}
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_row, task) for task in tasks]
    
    for future in tqdm(concurrent.futures.as_completed(futures), 
                      total=len(tasks), 
                      desc="Processing prompts"):
        idx, result = future.result()
        results[idx] = result

# Update DataFrame and save results
df['response_text'] = df.index.map(results)
df.to_csv(OUTPUT_CSV, index=False)

print("Processing complete. Results saved to", OUTPUT_CSV)

Processing prompts: 100%|██████████| 50/50 [06:40<00:00,  8.00s/it]

Processing complete. Results saved to /home/greg/issuebench/3_experiments/2_inference/completions/020925_gemma3-12b_completions.csv
